# Stable Diffusion CPU EP Performance Analysis — Float vs QDQ

**Objective:** Compare ORT CPU EP latency for float vs QDQ Stable Diffusion v1.5 components across AMD, Intel, and ARM devices.

**Components:** `text_encoder`, `unet`, `vae_decoder`, `vae_encoder`

**Devices:**
| Label | Platform | ORT Version | Iterations |
|-------|----------|-------------|------------|
| `amd` | AMD64 (HP AMD laptop) | 1.24.2 | 50 |
| `intel` | Intel64 (Surface Laptop) | 1.24.2 | 50 |
| `arm` | ARM64 (Surface ARM laptop) | 1.25.0-dev | 50 |

**Notes:**
- ARM uses a **dev build** of ORT (1.25.0-dev) — keep in mind when comparing
- All benchmarks use identical input shapes, 50 iterations, 5 warmup, seed 42
- Inputs: text_encoder `[1,77]`, unet `[1,4,64,64]+[1]+[1,77,768]`, vae_decoder `[1,4,64,64]`, vae_encoder `[1,3,512,512]`

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

# --- Load JSON data ---
DATA_DIR = Path(r"C:\Users\jambaykinley\OneDrive - Microsoft\Desktop\work-ARM\cpu-perf-sd")
DEVICE_FILES = {
    "amd": DATA_DIR / "amd.json",
    "intel": DATA_DIR / "surface_intel.json",
    "arm": DATA_DIR / "surface_arm.json",
}

DEVICE_ORDER = ["amd", "intel", "arm"]
COMPONENT_ORDER = ["text_encoder", "unet", "vae_decoder", "vae_encoder"]
METRIC_COLS = ["mean_ms", "std_ms", "min_ms", "max_ms", "p50_ms", "p95_ms", "p99_ms"]

frames = []
metadata = {}
for device, fpath in DEVICE_FILES.items():
    with open(fpath) as f:
        raw = json.load(f)
    metadata[device] = raw["metadata"]
    for model_type in ["float", "qdq"]:
        for component, data in raw["latency_ms"][model_type].items():
            row = {"device": device, "model_type": model_type, "component": component}
            for m in METRIC_COLS:
                row[m] = data[m]
            row["iterations"] = data["iterations"]
            row["warmup"] = data["warmup"]
            frames.append(row)
    print(f"  {device}: ORT {raw['metadata']['ort_version']}, "
          f"{raw['metadata']['platform']}, "
          f"total_elapsed={raw['metadata']['total_elapsed_s']:.0f}s")

df = pd.concat([pd.DataFrame([r]) for r in frames], ignore_index=True)

print(f"\nTotal rows: {len(df)}")
print(f"Devices: {list(df['device'].unique())}")
print(f"Model types: {list(df['model_type'].unique())}")
print(f"Components: {list(df['component'].unique())}")

# Save to CSV
out_path = Path("sd_cpu_perf_summary.csv")
df.to_csv(out_path, index=False)
print(f"\nSaved to {out_path}")

  amd: ORT 1.24.2, Windows-11-10.0.28020-SP0, total_elapsed=1005s
  intel: ORT 1.24.2, Windows-11-10.0.26200-SP0, total_elapsed=1545s
  arm: ORT 1.25.0.dev20260224003, Windows-10-10.0.26300-SP0, total_elapsed=1748s

Total rows: 24
Devices: ['amd', 'intel', 'arm']
Model types: ['float', 'qdq']
Components: ['text_encoder', 'unet', 'vae_decoder', 'vae_encoder']

Saved to sd_cpu_perf_summary.csv


## A. Per-Device: QDQ vs Float Latency

For each device, compare mean latency of QDQ vs float per component.

**Why QDQ is expected to be slower without fusion:**

QDQ is a graph annotation format — it inserts `QuantizeLinear` (Q) and `DequantizeLinear` (DQ) nodes around compute ops. Unless the runtime **fuses** `DQ → Op → Q` into an integer kernel (e.g., `QLinearConv`, `QLinearMatMul`), the model runs the same fp32 compute plus Q/DQ round-trips per layer:Without fusion, QDQ is **strictly slower** than float — the Q/DQ ops are pure overhead.



``````

Float path:  Input(fp32) → Conv/MatMul(fp32) → Output(fp32)QDQ path:    Input(fp32) → DQ(int8→fp32) → Conv/MatMul(fp32) → Q(fp32→int8) → ...

In [2]:
# Section A: QDQ vs Float per device
for device in DEVICE_ORDER:
    dev_df = df[df["device"] == device]

    float_df = dev_df[dev_df["model_type"] == "float"].set_index("component")["mean_ms"].rename("float_ms")
    qdq_df = dev_df[dev_df["model_type"] == "qdq"].set_index("component")["mean_ms"].rename("qdq_ms")

    tbl = pd.concat([float_df, qdq_df], axis=1).loc[COMPONENT_ORDER]
    tbl["qdq/float"] = tbl["qdq_ms"] / tbl["float_ms"]
    tbl["overhead_ms"] = tbl["qdq_ms"] - tbl["float_ms"]

    # Pipeline total (sum of all components — approximate single diffusion step)
    totals = tbl[["float_ms", "qdq_ms"]].sum()
    totals["qdq/float"] = totals["qdq_ms"] / totals["float_ms"]
    totals["overhead_ms"] = totals["qdq_ms"] - totals["float_ms"]
    tbl.loc["TOTAL"] = totals

    print(f"\n{'='*80}")
    print(f"  {device.upper()} — QDQ vs Float mean latency (ms)")
    print(f"  ORT {metadata[device]['ort_version']}, {metadata[device]['platform']}")
    print(f"{'='*80}")
    print(tbl.to_string(float_format=lambda x: f"{x:.1f}"))

print(f"\n{'='*80}")
print("  Summary: qdq/float ratio across devices")
print(f"{'='*80}")
summary_rows = []
for device in DEVICE_ORDER:
    dev_df = df[df["device"] == device]
    for comp in COMPONENT_ORDER:
        f_ms = dev_df[(dev_df["model_type"] == "float") & (dev_df["component"] == comp)]["mean_ms"].values[0]
        q_ms = dev_df[(dev_df["model_type"] == "qdq") & (dev_df["component"] == comp)]["mean_ms"].values[0]
        summary_rows.append({"device": device, "component": comp, "qdq/float": q_ms / f_ms})
summary = pd.DataFrame(summary_rows).pivot(index="component", columns="device", values="qdq/float")
summary = summary[DEVICE_ORDER].loc[COMPONENT_ORDER]
print(summary.to_string(float_format=lambda x: f"{x:.2f}"))


  AMD — QDQ vs Float mean latency (ms)
  ORT 1.24.2, Windows-11-10.0.28020-SP0
              float_ms  qdq_ms  qdq/float  overhead_ms
component                                             
text_encoder      24.5    61.5        2.5         37.0
unet            1405.3  2470.2        1.8       1064.9
vae_decoder     3689.1  5756.7        1.6       2067.7
vae_encoder     1961.2  2699.3        1.4        738.2
TOTAL           7080.0 10987.7        1.6       3907.7

  INTEL — QDQ vs Float mean latency (ms)
  ORT 1.24.2, Windows-11-10.0.26200-SP0
              float_ms  qdq_ms  qdq/float  overhead_ms
component                                             
text_encoder      53.6    81.7        1.5         28.1
unet            2038.8  3869.2        1.9       1830.4
vae_decoder     6160.3  8405.6        1.4       2245.3
vae_encoder     2865.4  4440.8        1.5       1575.4
TOTAL          11118.1 16797.3        1.5       5679.2

  ARM — QDQ vs Float mean latency (ms)
  ORT 1.25.0.dev20260224003,

### A — Observations

**1. QDQ is slower than float on every device and every component.** This is **expected** when QDQ nodes are not fused into integer compute kernels. Without fusion, `DequantizeLinear` and `QuantizeLinear` are pure overhead on top of the same fp32 compute.

**2. `text_encoder` has the worst qdq/float ratio (2.5–3.6×):** Shortest-running component (~22–53ms float), so the fixed per-layer Q/DQ overhead dominates. More quantized layers relative to total compute time = higher ratio.
- ARM: 3.62× (22ms → 82ms)
- AMD: 2.51× (24ms → 61ms)
- Intel: 1.52× (53ms → 82ms) — smallest penalty, but Intel float is already slow

**3. `vae_decoder` has the largest absolute overhead** (~2000ms extra on all devices) — most layers and longest runtime, so total Q/DQ cost accumulates.

**4. Overall pipeline qdq/float:**
- AMD: 1.55× (3.9s overhead per step)
- Intel: 1.51× (5.7s overhead — larger absolute cost due to slower baseline)
- ARM: 1.27× (3.8s overhead) — lowest ratio because ARM float is already slow (Q/DQ overhead is a smaller fraction)

**5. The qdq/float ratio is inversely correlated with component runtime.** Consistent with a fixed per-layer Q/DQ cost model: short components show higher ratios because the overhead is a larger fraction of total time.

**Next step:** Inspect the optimized graph to confirm which ops are fused vs unfused. If ORT supports `QLinearConv`/`QLinearMatMul` for the relevant shapes, enabling fusion would eliminate the Q/DQ overhead.

## B. Cross-Device Ratios

Compare absolute latency across devices using AMD as the reference (fastest device).
Shows device-specific kernel quality for each component under both float and QDQ.

In [3]:
# Section B: Cross-device ratios (AMD as reference)
for model_type in ["float", "qdq"]:
    sub = df[df["model_type"] == model_type]
    tbl = sub.pivot_table(index="component", columns="device", values="mean_ms", aggfunc="first")
    tbl = tbl[DEVICE_ORDER].loc[COMPONENT_ORDER]

    # Add ratios relative to AMD
    tbl["intel/amd"] = tbl["intel"] / tbl["amd"]
    tbl["arm/amd"] = tbl["arm"] / tbl["amd"]

    # Add total row
    totals = tbl[DEVICE_ORDER].sum()
    totals["intel/amd"] = totals["intel"] / totals["amd"]
    totals["arm/amd"] = totals["arm"] / totals["amd"]
    tbl.loc["TOTAL"] = totals

    print(f"\n{'='*90}")
    print(f"  B. Cross-device latency — {model_type.upper()} (ms, ratio vs AMD)")
    print(f"{'='*90}")
    print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))

# B.2: Does the QDQ penalty differ by device?
print(f"\n{'='*90}")
print(f"  B.2 QDQ overhead (qdq_ms - float_ms) per device (ms)")
print(f"{'='*90}")
overhead_rows = []
for device in DEVICE_ORDER:
    dev_df = df[df["device"] == device]
    for comp in COMPONENT_ORDER:
        f_ms = dev_df[(dev_df["model_type"] == "float") & (dev_df["component"] == comp)]["mean_ms"].values[0]
        q_ms = dev_df[(dev_df["model_type"] == "qdq") & (dev_df["component"] == comp)]["mean_ms"].values[0]
        overhead_rows.append({"device": device, "component": comp, "overhead_ms": q_ms - f_ms})
overhead = pd.DataFrame(overhead_rows).pivot(index="component", columns="device", values="overhead_ms")
overhead = overhead[DEVICE_ORDER].loc[COMPONENT_ORDER]
overhead.loc["TOTAL"] = overhead.sum()
print(overhead.to_string(float_format=lambda x: f"{x:.0f}"))


  B. Cross-device latency — FLOAT (ms, ratio vs AMD)
device           amd    intel      arm  intel/amd  arm/amd
component                                                 
text_encoder   24.51    53.65    22.63       2.19     0.92
unet         1405.28  2038.76  3124.34       1.45     2.22
vae_decoder  3689.09  6160.27  7316.75       1.67     1.98
vae_encoder  1961.15  2865.38  3336.64       1.46     1.70
TOTAL        7080.02 11118.06 13800.37       1.57     1.95

  B. Cross-device latency — QDQ (ms, ratio vs AMD)
device            amd    intel      arm  intel/amd  arm/amd
component                                                  
text_encoder    61.51    81.70    81.83       1.33     1.33
unet          2470.15  3869.18  3918.44       1.57     1.59
vae_decoder   5756.74  8405.59  9144.24       1.46     1.59
vae_encoder   2699.33  4440.82  4445.94       1.65     1.65
TOTAL        10987.72 16797.29 17590.46       1.53     1.60

  B.2 QDQ overhead (qdq_ms - float_ms) per device (ms)
devic

### B — Observations

**1. AMD is the fastest device for both float and QDQ.** Intel is 1.45–2.19× AMD, ARM is 0.92–2.22× AMD.

**2. ARM `text_encoder` float is faster than AMD** (0.92×) — the only case where ARM beats AMD. Likely a lightweight enough workload that ARM's core efficiency wins.

**3. Cross-device ratios are similar for float and QDQ** (Intel/AMD total: 1.57 float vs 1.53 qdq; ARM/AMD total: 1.95 float vs 1.60 qdq). This suggests the QDQ overhead scales proportionally with device speed — no device-specific kernel pathology.

**4. ARM's QDQ gap is smaller** (arm/amd_qdq = 1.60) **than its float gap** (arm/amd_float = 1.95). ARM narrows the gap under QDQ, possibly because the DequantizeLinear bottleneck (single-threaded scalar code) is equally slow everywhere and AMD's core-count advantage is less relevant.

**5. Absolute QDQ overhead by device:**
- Intel has the largest total overhead (5679ms) despite having a moderate qdq/float ratio (1.51×) — because its float baseline is slower than AMD
- AMD and ARM have similar total overhead (~3800–3900ms) despite very different absolute latencies

## C. Measurement Variance

Compare coefficient of variation (`std_ms / mean_ms`) across devices. Higher CV suggests less stable measurements (thermal throttling, background processes, memory pressure).

In [4]:
# Section C: Measurement variance — coefficient of variation
df["cv"] = df["std_ms"] / df["mean_ms"]

cv_tbl = df.pivot_table(
    index=["model_type", "component"],
    columns="device",
    values="cv",
    aggfunc="first"
)[DEVICE_ORDER]

print(f"{'='*80}")
print(f"  C. Coefficient of Variation (std/mean) — lower is more stable")
print(f"{'='*80}")
print(cv_tbl.to_string(float_format=lambda x: f"{x:.4f}"))

# Summary per device
print(f"\n  Per-device CV summary:")
for dev in DEVICE_ORDER:
    vals = cv_tbl[dev]
    print(f"    {dev}: mean={vals.mean():.4f}  max={vals.max():.4f}  "
          f"{'⚠ HIGH' if vals.max() > 0.10 else 'OK'}")

# Also show p95/p50 spread as another stability indicator
spread_tbl = df.copy()
spread_tbl["p95/p50"] = spread_tbl["p95_ms"] / spread_tbl["p50_ms"]
spread = spread_tbl.pivot_table(
    index=["model_type", "component"],
    columns="device",
    values="p95/p50",
    aggfunc="first"
)[DEVICE_ORDER]

print(f"\n{'='*80}")
print(f"  C.2 p95/p50 ratio — tail latency indicator (closer to 1.0 = stable)")
print(f"{'='*80}")
print(spread.to_string(float_format=lambda x: f"{x:.3f}"))

  C. Coefficient of Variation (std/mean) — lower is more stable
device                     amd  intel    arm
model_type component                        
float      text_encoder 0.0327 0.3272 0.2947
           unet         0.0806 0.0833 0.1635
           vae_decoder  0.0622 0.1587 0.0329
           vae_encoder  0.0458 0.1189 0.0352
qdq        text_encoder 0.0324 0.1919 0.1174
           unet         0.0290 0.0511 0.0792
           vae_decoder  0.0514 0.0280 0.0487
           vae_encoder  0.0534 0.1434 0.0699

  Per-device CV summary:
    amd: mean=0.0484  max=0.0806  OK
    intel: mean=0.1378  max=0.3272  ⚠ HIGH
    arm: mean=0.1052  max=0.2947  ⚠ HIGH

  C.2 p95/p50 ratio — tail latency indicator (closer to 1.0 = stable)
device                    amd  intel   arm
model_type component                      
float      text_encoder 1.068  1.968 1.647
           unet         1.227  1.157 1.412
           vae_decoder  1.110  1.457 1.061
           vae_encoder  1.075  1.249 1.041
qdq       

### C — Observations

**1. AMD measurements are stable** — max CV 8.06% (float unet), all others <7%. Reliable data.

**2. Intel and ARM `text_encoder` float have very high variance:**
- Intel float text_encoder: CV 32.7%, p95/p50 = 1.97 (tail latency nearly 2× median)
- ARM float text_encoder: CV 29.5%, p95/p50 = 1.65
- These are lightweight components (~25–55ms) where OS scheduling jitter dominates

**3. QDQ measurements tend to be more stable than float** on Intel and ARM:
- Intel text_encoder: CV drops from 32.7% (float) → 19.2% (qdq)
- ARM text_encoder: CV drops from 29.5% → 11.7%
- Likely because QDQ adds deterministic compute overhead that dwarfs scheduling noise

**4. Longer-running components are more stable** (vae_decoder CV < 7% across all devices) — measurement noise is amortized over longer runtimes.

**5. All data is usable for relative comparisons** despite text_encoder noise, since the same variance affects both float and QDQ measurements on a given device.

## D. Executive Summary Table

Single wide table showing mean latency per device × model type, qdq/float ratios, and cross-device ratios.

In [5]:
# Section D: Executive summary — wide table
rows = []
for comp in COMPONENT_ORDER:
    row = {"component": comp}
    for dev in DEVICE_ORDER:
        for mt in ["float", "qdq"]:
            val = df[(df["device"] == dev) & (df["model_type"] == mt) & (df["component"] == comp)]["mean_ms"].values[0]
            row[f"{dev}_{mt}"] = val
        row[f"{dev}_qdq/float"] = row[f"{dev}_qdq"] / row[f"{dev}_float"]
    # Cross-device ratios (float)
    row["intel/amd_float"] = row["intel_float"] / row["amd_float"]
    row["arm/amd_float"] = row["arm_float"] / row["amd_float"]
    # Cross-device ratios (qdq)
    row["intel/amd_qdq"] = row["intel_qdq"] / row["amd_qdq"]
    row["arm/amd_qdq"] = row["arm_qdq"] / row["amd_qdq"]
    rows.append(row)

summary = pd.DataFrame(rows).set_index("component")

# Add TOTAL row
total = {}
for col in summary.columns:
    if "qdq/float" in col or "/" in col:
        continue  # Recompute ratios for totals
    total[col] = summary[col].sum()
for dev in DEVICE_ORDER:
    total[f"{dev}_qdq/float"] = total[f"{dev}_qdq"] / total[f"{dev}_float"]
total["intel/amd_float"] = total["intel_float"] / total["amd_float"]
total["arm/amd_float"] = total["arm_float"] / total["amd_float"]
total["intel/amd_qdq"] = total["intel_qdq"] / total["amd_qdq"]
total["arm/amd_qdq"] = total["arm_qdq"] / total["amd_qdq"]
summary.loc["TOTAL"] = total

# Display with grouped column headers
print(f"{'='*160}")
print(f"  D. Executive Summary — Mean Latency (ms) + Ratios")
print(f"{'='*160}")

# Reorder columns for readability
col_order = []
for dev in DEVICE_ORDER:
    col_order += [f"{dev}_float", f"{dev}_qdq", f"{dev}_qdq/float"]
col_order += ["intel/amd_float", "intel/amd_qdq", "arm/amd_float", "arm/amd_qdq"]
summary = summary[col_order]

print(summary.to_string(float_format=lambda x: f"{x:.2f}"))

  D. Executive Summary — Mean Latency (ms) + Ratios
              amd_float  amd_qdq  amd_qdq/float  intel_float  intel_qdq  intel_qdq/float  arm_float  arm_qdq  arm_qdq/float  intel/amd_float  intel/amd_qdq  arm/amd_float  arm/amd_qdq
component                                                                                                                                                                              
text_encoder      24.51    61.51           2.51        53.65      81.70             1.52      22.63    81.83           3.62             2.19           1.33           0.92         1.33
unet            1405.28  2470.15           1.76      2038.76    3869.18             1.90    3124.34  3918.44           1.25             1.45           1.57           2.22         1.59
vae_decoder     3689.09  5756.74           1.56      6160.27    8405.59             1.36    7316.75  9144.24           1.25             1.67           1.46           1.98         1.59
vae_encoder     1961.15  269

## E. Why QDQ Fusion Fails — uint16 Activation Quantization

### Model quantization scheme

| Component | Data type | Granularity |
|-----------|-----------|-------------|
| **Weights** (Conv, Gemm, MatMul) | `uint8` | per-tensor |
| **Bias** | `int32` | — |
| **Activations** | `uint16` | per-tensor |

### Root cause: ORT CPU EP integer kernels only support int8/uint8

The QDQ fusion selectors (`ConvSelector`, `MatMulSelector`, `GemmSelector`) in `onnxruntime/core/optimizer/selectors_actions/qdq_selectors.cc` validate that **both input activations and output activations** are `int8` or `uint8`. `uint16` is rejected, so no `DQ → Op → Q` pattern matches.

The fused kernels (`QLinearConv`, `QLinearMatMul`) use MLAS int8 GEMM routines (`MlasGemm(U8S8)`, `MlasGemm(U8U8)`) that only accept 8-bit operands. There is no `MlasGemm(U16U8)` — CPUs have no native uint16×uint8 multiply-accumulate SIMD instructions (unlike int8 `VPMADDUBSW`/`VPDPBUSD` on x86 or `SMMLA`/`UMMLA` on ARM).

### What actually executes

```
DQ(uint16 act → fp32) → DQ(uint8 wt → fp32) → Conv/MatMul(fp32) → Q(fp32 → uint16)
│                        │                      │                    │
│ overhead               │ overhead              │ same as float      │ overhead
```

Every Q and DQ node runs as a **standalone fp32 cast** — no fusion, no integer compute. The model performs identical fp32 arithmetic as the float model, plus the Q/DQ round-trip cost at every layer boundary. This fully explains the 1.25–3.6× overhead measured in Section A.

### Available QDQ fusion targets on CPU EP

| Fused Op | Activation types | Weight types | Available? |
|----------|-----------------|--------------|-----------|
| `QLinearConv` | int8, uint8 | int8, uint8 | ✅ — but needs uint8 activations |
| `QLinearMatMul` | int8, uint8 | int8, uint8 | ✅ — but needs uint8 activations |
| `MatMulIntegerToFloat` | int8, uint8 | int8, uint8 | ✅ — no output Q needed |
| `QLinearSigmoid` | int8, uint8 | — | ✅ |
| `QLinearLeakyRelu` | int8, uint8 | — | ✅ |
| `MatMulNBits` | N/A (weight-only) | 4-bit block | ✅ — no activation Q/DQ |

**None support uint16.** The model's uint16 activations make all QDQ nodes unfusible.

### Alternatives for SD on CPU EP

| Strategy | Pros | Cons |
|----------|------|------|
| **Re-quantize with uint8 activations** | Enables `QLinearConv`/`QLinearMatMul` fusion, potential speedup | May degrade image quality (8-bit activation grid too coarse for diffusion) |
| **Weight-only quantization** (no activation Q) | No Q/DQ overhead on activations; `MatMulNBits`-style for linear layers | Only helps MatMul/Gemm, not Conv; requires re-export |
| **Keep float model** | No overhead, best accuracy | No compression benefit |
| **Target a different EP** (QNN, TensorRT) | These EPs may have uint16 integer kernels or custom fusion rules | Not CPU EP |

In [ ]:
# Section E: Verify — dump optimized graph op counts
# This confirms whether QLinearConv/QLinearMatMul fusion happened
# Uncomment and set the path to one of the QDQ model files to run

# import onnxruntime as ort
# import onnx
#
# qdq_model_path = r"C:\path\to\sd\qdq\unet\model.onnx"  # <-- SET THIS
#
# so = ort.SessionOptions()
# so.optimized_model_filepath = "unet_optimized.onnx"
# so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
# sess = ort.InferenceSession(qdq_model_path, so, providers=["CPUExecutionProvider"])
#
# model = onnx.load("unet_optimized.onnx")
# op_counts = {}
# for node in model.graph.node:
#     op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1
#
# print("Optimized graph op counts:")
# print("-" * 40)
# for op, count in sorted(op_counts.items(), key=lambda x: -x[1]):
#     print(f"  {op:30s} {count}")
#
# # Check for fusion
# fused_ops = {"QLinearConv", "QLinearMatMul", "MatMulIntegerToFloat", "QLinearSigmoid"}
# remaining_qdq = op_counts.get("DequantizeLinear", 0) + op_counts.get("QuantizeLinear", 0)
# fused_count = sum(op_counts.get(op, 0) for op in fused_ops)
# print(f"\n  Remaining Q/DQ nodes: {remaining_qdq}")
# print(f"  Fused integer ops:    {fused_count}")
# if remaining_qdq > 0 and fused_count == 0:
#     print("  → NO fusion occurred — all Q/DQ nodes are unfused overhead")

## F. Q/DQ Overhead Efficiency Analysis

Given that QDQ fusion is not happening (due to uint16 activations), all `QuantizeLinear` and `DequantizeLinear` nodes run as standalone element-wise ops. The question is: **is the observed overhead consistent with an efficient (SIMD-vectorized, memory-bandwidth-bound) implementation, or does it suggest room for optimization?**

### What Q/DQ nodes actually compute

| Op | Formula | Bytes/element | Compute |
|----|---------|--------------|---------|
| `DequantizeLinear` (uint16→fp32) | `out = (in - zp) * scale` | read 2B + write 4B = **6 B** | 1 sub + 1 mul |
| `QuantizeLinear` (fp32→uint16) | `out = clamp(round(in / scale) + zp)` | read 4B + write 2B = **6 B** | 1 div + 1 round + 1 add + 1 clamp |
| `DequantizeLinear` (uint8→fp32, weights) | `out = (in - zp) * scale` | read 1B + write 4B = **5 B** | 1 sub + 1 mul |

With per-tensor quantization (scalar scale/zp), these are **trivially memory-bandwidth-bound** — ~1–4 FLOPs per element, no reductions, no cross-element dependencies. A well-optimized implementation should run at near-peak DRAM bandwidth.

### Efficiency indicators

**1. Overhead as fraction of float compute:** For compute-heavy ops like Conv and MatMul (which are O(n·k²·c) or O(n·m·k) compute), the Q/DQ overhead (O(n) elementwise) should be a **small fraction** of the compute time. If it's large, either there are many Q/DQ nodes or they're inefficient.

**2. Cross-device overhead scaling:** If Q/DQ runs at memory bandwidth, devices with similar DRAM bandwidth should show similar absolute overhead. If the overhead tracks CPU compute speed instead, that suggests the implementation is compute-bound (scalar, not SIMD-vectorized).

**3. Overhead ratio vs float ratio:**
- `overhead_ratio ≈ float_ratio` → Q/DQ is compute-bound (scales with CPU speed) → **suboptimal, room for SIMD optimization**
- `overhead_ratio ≈ 1.0` → Q/DQ is memory-BW-bound → **already efficient**
- `overhead_ratio > float_ratio` → worse than compute-bound → **pathological** (cache thrashing, thread issues)

In [6]:
# Section F: Q/DQ overhead efficiency analysis

print("="*100)
print("  F. Q/DQ Overhead Efficiency Analysis")
print("="*100)

# F.1: Observed QDQ overhead per component
print("\n  F.1 Observed QDQ overhead per component (ms)")
print("  " + "-"*96)

overhead_data = {}
for device in DEVICE_ORDER:
    dev_df = df[df["device"] == device]
    overhead_data[device] = {}
    for comp in COMPONENT_ORDER:
        f_ms = dev_df[(dev_df["model_type"] == "float") & (dev_df["component"] == comp)]["mean_ms"].values[0]
        q_ms = dev_df[(dev_df["model_type"] == "qdq") & (dev_df["component"] == comp)]["mean_ms"].values[0]
        overhead_data[device][comp] = q_ms - f_ms

overhead_df = pd.DataFrame(overhead_data)
overhead_df.loc["TOTAL"] = overhead_df.sum()
print(overhead_df.to_string(float_format=lambda x: f"{x:.1f}"))

# F.2: Overhead as fraction of float compute time
print(f"\n  F.2 Overhead as fraction of float compute time (overhead_ms / float_ms)")
print("  " + "-"*96)
frac_data = {}
for device in DEVICE_ORDER:
    dev_df = df[df["device"] == device]
    frac_data[device] = {}
    for comp in COMPONENT_ORDER:
        f_ms = dev_df[(dev_df["model_type"] == "float") & (dev_df["component"] == comp)]["mean_ms"].values[0]
        frac_data[device][comp] = overhead_data[device][comp] / f_ms
frac_df = pd.DataFrame(frac_data)
print(frac_df.to_string(float_format=lambda x: f"{x:.2f}"))

print("""
  Interpretation:
  - overhead/float = 0.25 means Q/DQ adds 25% on top of float compute
  - overhead/float = 2.60 means Q/DQ takes 2.6x as long as the actual compute

  For compute-heavy ops (Conv, MatMul), Q/DQ is O(n) elementwise while
  the compute is O(n*k^2*c) or O(n*m*k). Overhead/float should be SMALL
  (<0.1-0.2) for an efficient Q/DQ implementation on compute-heavy models.
""")

# F.3: Cross-device overhead ratios
print(f"  F.3 Cross-device overhead ratios (vs AMD)")
print("  " + "-"*96)
ratio_data = {}
for comp in COMPONENT_ORDER:
    ratio_data[comp] = {
        "intel/amd": overhead_data["intel"][comp] / overhead_data["amd"][comp],
        "arm/amd": overhead_data["arm"][comp] / overhead_data["amd"][comp],
    }
ratio_df = pd.DataFrame(ratio_data).T
print(ratio_df.to_string(float_format=lambda x: f"{x:.2f}"))

print("""
  If Q/DQ is memory-bandwidth-bound:
    - Devices with similar DRAM BW should show ratio ~ 1.0
    - AMD DDR5 ~40-50 GB/s, Intel LPDDR5x ~50-70 GB/s, ARM LPDDR5x ~50-70 GB/s
    - Expected: intel/amd ~ 0.7-1.0, arm/amd ~ 0.7-1.0

  If Q/DQ is compute-bound (scalar, no SIMD):
    - Ratios would track CPU compute speed differences
    - Expected: intel/amd ~ 1.5-2.0, arm/amd ~ 1.5-2.0 (matching float ratios)
""")

# F.4: Compare overhead ratios to float compute ratios
print(f"  F.4 Overhead ratio vs Float compute ratio")
print("  " + "-"*96)
comparison_rows = []
for comp in COMPONENT_ORDER:
    dev_df_amd = df[df["device"] == "amd"]
    dev_df_intel = df[df["device"] == "intel"]
    dev_df_arm = df[df["device"] == "arm"]

    f_amd = dev_df_amd[(dev_df_amd["model_type"] == "float") & (dev_df_amd["component"] == comp)]["mean_ms"].values[0]
    f_intel = dev_df_intel[(dev_df_intel["model_type"] == "float") & (dev_df_intel["component"] == comp)]["mean_ms"].values[0]
    f_arm = dev_df_arm[(dev_df_arm["model_type"] == "float") & (dev_df_arm["component"] == comp)]["mean_ms"].values[0]

    comparison_rows.append({
        "component": comp,
        "intel/amd_float": f_intel / f_amd,
        "intel/amd_overhead": overhead_data["intel"][comp] / overhead_data["amd"][comp],
        "arm/amd_float": f_arm / f_amd,
        "arm/amd_overhead": overhead_data["arm"][comp] / overhead_data["amd"][comp],
    })
comp_df = pd.DataFrame(comparison_rows).set_index("component")
print(comp_df.to_string(float_format=lambda x: f"{x:.2f}"))

print("""
  Key:
  - If overhead ratios ~ float ratios -> Q/DQ is compute-bound (scales with CPU speed)
    -> Suggests SCALAR implementation, no SIMD -> room for optimization
  - If overhead ratios ~ 1.0 (or track memory BW) -> Q/DQ is BW-bound
    -> Already efficient, limited room for improvement
  - If overhead ratios > float ratios -> worse than compute-bound
    -> Possible cache thrashing, thread contention, or suboptimal memory patterns
""")

  F. Q/DQ Overhead Efficiency Analysis

  F.1 Observed QDQ overhead per component (ms)
  ------------------------------------------------------------------------------------------------
                amd  intel    arm
text_encoder   37.0   28.1   59.2
unet         1064.9 1830.4  794.1
vae_decoder  2067.7 2245.3 1827.5
vae_encoder   738.2 1575.4 1109.3
TOTAL        3907.7 5679.2 3790.1

  F.2 Overhead as fraction of float compute time (overhead_ms / float_ms)
  ------------------------------------------------------------------------------------------------
              amd  intel  arm
text_encoder 1.51   0.52 2.62
unet         0.76   0.90 0.25
vae_decoder  0.56   0.36 0.25
vae_encoder  0.38   0.55 0.33

  Interpretation:
  - overhead/float = 0.25 means Q/DQ adds 25% on top of float compute
  - overhead/float = 2.60 means Q/DQ takes 2.6x as long as the actual compute

  For compute-heavy ops (Conv, MatMul), Q/DQ is O(n) elementwise while
  the compute is O(n*k^2*c) or O(n*m*k). Overhe

### F — Observations

**1. Overhead is significant but must be interpreted in context of Q/DQ volume:**
- text_encoder: 0.52–2.62× of float compute
- unet/vae: 0.25–0.90× of float compute
- Every compute op in the model is wrapped: `DQ(activation) + DQ(weight) [+ DQ(bias)] + Op + Q(output)` = 3–4 Q/DQ nodes per compute op
- A typical SD v1.5 unet has ~300–500 compute ops → **1000–2000 Q/DQ nodes** total
- Each processes a full tensor (activations can be large: e.g., `[1,320,64,64]` = 1.3M elements)
- The high overhead fractions may be **reasonable** given this volume, not necessarily indicating a slow kernel

**2. To properly assess kernel efficiency, we need:**
- Total Q/DQ node count per component (from the ONNX graph)
- Total elements processed across all Q/DQ nodes
- Then: `achieved_bandwidth = bytes_processed / overhead_ms` vs peak DRAM BW
- Without this, we cannot distinguish "lots of efficient Q/DQ work" from "fewer inefficient Q/DQ ops"

**3. Cross-device overhead ratios do NOT follow a clean pattern:**
- Intel overhead varies: 0.76× AMD for text_encoder but 2.13× AMD for vae_encoder
- ARM: 0.75× for unet (faster than AMD!) but 1.60× for text_encoder
- This inconsistency suggests the overhead is influenced by **memory allocation patterns** (malloc/free for each Q/DQ output tensor), cache effects, and thread scheduling — not purely compute or bandwidth.

**4. F.4 shows overhead ratios do NOT track float ratios:**
- text_encoder: intel/amd float=2.19 but overhead=0.76 (opposite direction!)
- unet: arm/amd float=2.22 but overhead=0.75 (opposite direction!)
- The overhead has **different scaling characteristics** than the core compute, suggesting the bottleneck is something other than raw arithmetic throughput.

**5. Regardless of kernel efficiency, the fundamental issue remains:**
- Without fusion, every Q/DQ node allocates a full output tensor and performs a full tensor copy+cast
- Even a perfectly optimized Q/DQ kernel still pays the memory traffic cost of materializing intermediate fp32 tensors at every layer boundary
- The only way to eliminate this overhead is fusion (`DQ→Op→Q` → integer kernel) or removing Q/DQ entirely (weight-only quantization / float model)

**Next step:** Dump the ONNX graph to count Q/DQ nodes and sum their tensor sizes per component. This enables a proper achieved-bandwidth calculation to determine if the Q/DQ kernel itself is efficient or has room for improvement.

## G. ONNX Graph Analysis — Op Counts, Q/DQ Volume, and Parameter Count

Load the float and QDQ models to:
1. Count ops by type (confirm Q/DQ are unfused)
2. Count total Q/DQ nodes and estimate total elements they process
3. Count parameters (weights) per component
4. Compute achieved bandwidth of Q/DQ ops vs peak DRAM BW

In [2]:
import onnx
from collections import Counter

MODEL_ROOT = Path(r"C:\Users\jambaykinley\Downloads\sd")
MODEL_TYPES = ["float", "qdq"]

def get_elem_type_name(elem_type):
    """Map ONNX TensorProto.DataType enum to readable string."""
    names = {
        1: "float32", 2: "uint8", 3: "int8", 4: "uint16", 5: "int16",
        6: "int32", 7: "int64", 9: "string", 10: "float16", 11: "double",
        12: "uint32", 13: "uint64", 16: "bfloat16",
    }
    return names.get(elem_type, f"unknown({elem_type})")

def get_tensor_size(shape_dims):
    """Compute number of elements from shape dims. Returns None if any dim is dynamic."""
    if not shape_dims:
        return None
    size = 1
    for d in shape_dims:
        if d <= 0:  # dynamic or unknown
            return None
        size *= d
    return size

def bytes_per_element(elem_type):
    """Bytes per element for ONNX data types."""
    bpe = {1: 4, 2: 1, 3: 1, 4: 2, 5: 2, 6: 4, 7: 8, 10: 2, 11: 8, 12: 4, 13: 8, 16: 2}
    return bpe.get(elem_type, 4)

def analyze_model(model_path):
    """Analyze an ONNX model graph without loading external data."""
    model = onnx.load(str(model_path), load_external_data=False)
    graph = model.graph

    # --- Op counts ---
    op_counts = Counter(node.op_type for node in graph.node)

    # --- Build tensor info map from graph inputs, outputs, value_info, and initializers ---
    tensor_info = {}  # name -> (elem_type, shape_dims)

    for vi in list(graph.input) + list(graph.output) + list(graph.value_info):
        t = vi.type.tensor_type
        if t.HasField("shape"):
            dims = [d.dim_value for d in t.shape.dim]
        else:
            dims = []
        tensor_info[vi.name] = (t.elem_type, dims)

    for init in graph.initializer:
        tensor_info[init.name] = (init.data_type, list(init.dims))

    # --- Count parameters (initializer weights) ---
    total_params = 0
    param_bytes = 0
    for init in graph.initializer:
        n = get_tensor_size(list(init.dims))
        if n:
            total_params += n
            param_bytes += n * bytes_per_element(init.data_type)

    # --- Analyze Q/DQ nodes ---
    qdq_ops = ["QuantizeLinear", "DequantizeLinear"]
    qdq_nodes = [n for n in graph.node if n.op_type in qdq_ops]

    qdq_total_elements = 0
    qdq_total_bytes = 0  # read + write bytes
    qdq_details = {"QuantizeLinear": {"count": 0, "elements": 0, "bytes": 0},
                   "DequantizeLinear": {"count": 0, "elements": 0, "bytes": 0}}

    for node in qdq_nodes:
        # Input tensor (the data being quantized/dequantized)
        input_name = node.input[0]
        output_name = node.output[0]

        in_info = tensor_info.get(input_name)
        out_info = tensor_info.get(output_name)

        # Use whichever has shape info
        if in_info and in_info[1]:
            n_elem = get_tensor_size(in_info[1])
            in_bpe = bytes_per_element(in_info[0])
        else:
            n_elem = None
            in_bpe = 4

        if out_info and out_info[1]:
            if n_elem is None:
                n_elem = get_tensor_size(out_info[1])
            out_bpe = bytes_per_element(out_info[0])
        else:
            out_bpe = 4

        if n_elem:
            qdq_total_elements += n_elem
            rw_bytes = n_elem * (in_bpe + out_bpe)
            qdq_total_bytes += rw_bytes
            qdq_details[node.op_type]["count"] += 1
            qdq_details[node.op_type]["elements"] += n_elem
            qdq_details[node.op_type]["bytes"] += rw_bytes
        else:
            qdq_details[node.op_type]["count"] += 1

    # --- Activation data types (from DequantizeLinear inputs that are NOT initializers) ---
    init_names = {init.name for init in graph.initializer}
    act_dtypes = Counter()
    weight_dtypes = Counter()
    for node in graph.node:
        if node.op_type == "DequantizeLinear":
            inp = node.input[0]
            info = tensor_info.get(inp)
            dtype_str = get_elem_type_name(info[0]) if info else "unknown"
            if inp in init_names:
                weight_dtypes[dtype_str] += 1
            else:
                act_dtypes[dtype_str] += 1
        elif node.op_type == "QuantizeLinear":
            out = node.output[0]
            info = tensor_info.get(out)
            dtype_str = get_elem_type_name(info[0]) if info else "unknown"
            act_dtypes[dtype_str] += 1

    return {
        "op_counts": op_counts,
        "total_nodes": len(graph.node),
        "total_params": total_params,
        "param_bytes": param_bytes,
        "qdq_node_count": len(qdq_nodes),
        "qdq_total_elements": qdq_total_elements,
        "qdq_total_bytes": qdq_total_bytes,
        "qdq_details": qdq_details,
        "act_dtypes": dict(act_dtypes),
        "weight_dtypes": dict(weight_dtypes),
        "n_initializers": len(graph.initializer),
    }

# --- Run analysis ---
results = {}
for model_type in MODEL_TYPES:
    results[model_type] = {}
    for comp in COMPONENT_ORDER:
        model_path = MODEL_ROOT / model_type / comp / "model.onnx"
        if model_path.exists():
            print(f"Loading {model_type}/{comp}...", end=" ")
            info = analyze_model(model_path)
            results[model_type][comp] = info
            print(f"{info['total_nodes']} nodes, {info['total_params']:,} params, "
                  f"{info['qdq_node_count']} Q/DQ nodes")
        else:
            print(f"  MISSING: {model_path}")

print("\nDone loading all models.")

Loading float/text_encoder... 1104 nodes, 123,060,480 params, 0 Q/DQ nodes
Loading float/unet... 4293 nodes, 859,520,964 params, 0 Q/DQ nodes
Loading float/vae_decoder... 520 nodes, 49,490,199 params, 0 Q/DQ nodes
Loading float/vae_encoder... 502 nodes, 34,163,664 params, 0 Q/DQ nodes
Loading qdq/text_encoder... 1417 nodes, 123,132,125 params, 1013 Q/DQ nodes
Loading qdq/unet... 5482 nodes, 859,526,206 params, 3937 Q/DQ nodes
Loading qdq/vae_decoder... 1061 nodes, 49,492,346 params, 775 Q/DQ nodes
Loading qdq/vae_encoder... 823 nodes, 34,181,635 params, 601 Q/DQ nodes

Done loading all models.


In [3]:
# G.1: Op count comparison — float vs QDQ
print("="*110)
print("  G.1 Op Counts — Float vs QDQ")
print("="*110)

for comp in COMPONENT_ORDER:
    float_info = results["float"][comp]
    qdq_info = results["qdq"][comp]

    print(f"\n  --- {comp} ---")
    print(f"  {'Op Type':35s} {'Float':>8s} {'QDQ':>8s} {'Diff':>8s}")
    print(f"  {'-'*35} {'-'*8} {'-'*8} {'-'*8}")

    all_ops = sorted(set(list(float_info["op_counts"].keys()) + list(qdq_info["op_counts"].keys())))
    for op in all_ops:
        fc = float_info["op_counts"].get(op, 0)
        qc = qdq_info["op_counts"].get(op, 0)
        diff = qc - fc
        diff_str = f"+{diff}" if diff > 0 else str(diff) if diff != 0 else ""
        print(f"  {op:35s} {fc:8d} {qc:8d} {diff_str:>8s}")

    print(f"  {'TOTAL':35s} {float_info['total_nodes']:8d} {qdq_info['total_nodes']:8d} "
          f"+{qdq_info['total_nodes'] - float_info['total_nodes']}")

  G.1 Op Counts — Float vs QDQ

  --- text_encoder ---
  Op Type                                Float      QDQ     Diff
  ----------------------------------- -------- -------- --------
  Add                                      112       37      -75
  ArgMax                                     1        1         
  Cast                                       5        0       -5
  Concat                                    54        0      -54
  Constant                                 352        0     -352
  ConstantOfShape                            2        0       -2
  DequantizeLinear                           0      611     +611
  Equal                                      1        0       -1
  Expand                                     1        0       -1
  Flatten                                    1        1         
  Gather                                    57        2      -55
  Gemm                                       0       72      +72
  LayerNormalization               

In [4]:
# G.2: Parameter counts — float vs QDQ
print("="*110)
print("  G.2 Parameter Counts — Float vs QDQ")
print("="*110)

param_rows = []
for comp in COMPONENT_ORDER:
    float_info = results["float"][comp]
    qdq_info = results["qdq"][comp]
    param_rows.append({
        "component": comp,
        "float_params": float_info["total_params"],
        "float_MB": float_info["param_bytes"] / 1e6,
        "qdq_params": qdq_info["total_params"],
        "qdq_MB": qdq_info["param_bytes"] / 1e6,
        "qdq_n_initializers": qdq_info["n_initializers"],
        "float_n_initializers": float_info["n_initializers"],
    })
param_df = pd.DataFrame(param_rows).set_index("component")

# Add size ratio
param_df["size_ratio"] = param_df["qdq_MB"] / param_df["float_MB"]

print(param_df.to_string(float_format=lambda x: f"{x:.1f}" if x < 1e6 else f"{x:,.0f}"))

# Total
total_f = param_df["float_MB"].sum()
total_q = param_df["qdq_MB"].sum()
print(f"\n  TOTAL: Float {total_f:.1f} MB, QDQ {total_q:.1f} MB, ratio {total_q/total_f:.2f}×")

  G.2 Parameter Counts — Float vs QDQ
              float_params  float_MB  qdq_params  qdq_MB  qdq_n_initializers  float_n_initializers  size_ratio
component                                                                                                     
text_encoder     123060480     492.2   123132125   161.5                1187                   196         0.3
unet             859520964    3438.1   859526206   860.5                4674                   686         0.3
vae_decoder       49490199     198.0    49492346    49.6                1063                   140         0.3
vae_encoder       34163664     136.7    34181635    34.3                 828                   108         0.3

  TOTAL: Float 4264.9 MB, QDQ 1105.8 MB, ratio 0.26×


In [5]:
# G.3: Q/DQ node volume and bandwidth analysis
print("="*110)
print("  G.3 Q/DQ Node Volume and Achieved Bandwidth")
print("="*110)

for comp in COMPONENT_ORDER:
    qdq_info = results["qdq"][comp]
    det = qdq_info["qdq_details"]

    print(f"\n  --- {comp} ---")
    print(f"  Total graph nodes:    {qdq_info['total_nodes']}")
    print(f"  Q/DQ nodes:           {qdq_info['qdq_node_count']}  "
          f"({qdq_info['qdq_node_count']/qdq_info['total_nodes']*100:.0f}% of all nodes)")
    print(f"    DequantizeLinear:   {det['DequantizeLinear']['count']}")
    print(f"    QuantizeLinear:     {det['QuantizeLinear']['count']}")
    print(f"  Total elements:       {qdq_info['qdq_total_elements']:,.0f}")
    print(f"  Total read+write:     {qdq_info['qdq_total_bytes']:,.0f} bytes "
          f"({qdq_info['qdq_total_bytes']/1e9:.3f} GB)")

    # Activation + weight dtype breakdown
    print(f"  Activation Q/DQ dtypes: {qdq_info['act_dtypes']}")
    print(f"  Weight DQ dtypes:       {qdq_info['weight_dtypes']}")

    # Achieved bandwidth using overhead from Section F
    for device in DEVICE_ORDER:
        dev_df_local = df[df["device"] == device]
        f_ms = dev_df_local[(dev_df_local["model_type"] == "float") & (dev_df_local["component"] == comp)]["mean_ms"].values[0]
        q_ms = dev_df_local[(dev_df_local["model_type"] == "qdq") & (dev_df_local["component"] == comp)]["mean_ms"].values[0]
        overhead_ms = q_ms - f_ms
        if overhead_ms > 0 and qdq_info["qdq_total_bytes"] > 0:
            bw_gb_s = (qdq_info["qdq_total_bytes"] / 1e9) / (overhead_ms / 1000)
            print(f"  Achieved BW ({device:5s}): {bw_gb_s:.1f} GB/s  "
                  f"(overhead={overhead_ms:.0f}ms, "
                  f"data={qdq_info['qdq_total_bytes']/1e6:.0f}MB)")

# Summary table
print(f"\n{'='*110}")
print(f"  G.3 Summary — Q/DQ volume per component")
print(f"{'='*110}")
vol_rows = []
for comp in COMPONENT_ORDER:
    qi = results["qdq"][comp]
    fi = results["float"][comp]
    vol_rows.append({
        "component": comp,
        "float_nodes": fi["total_nodes"],
        "qdq_nodes_total": qi["total_nodes"],
        "Q_nodes": qi["qdq_details"]["QuantizeLinear"]["count"],
        "DQ_nodes": qi["qdq_details"]["DequantizeLinear"]["count"],
        "Q+DQ_pct": qi["qdq_node_count"] / qi["total_nodes"] * 100,
        "qdq_data_MB": qi["qdq_total_bytes"] / 1e6,
    })
vol_df = pd.DataFrame(vol_rows).set_index("component")
vol_df.loc["TOTAL"] = vol_df.sum()
# Fix percentage for total
total_qdq = sum(results["qdq"][c]["qdq_node_count"] for c in COMPONENT_ORDER)
total_nodes = sum(results["qdq"][c]["total_nodes"] for c in COMPONENT_ORDER)
vol_df.loc["TOTAL", "Q+DQ_pct"] = total_qdq / total_nodes * 100
print(vol_df.to_string(float_format=lambda x: f"{x:.1f}"))

  G.3 Q/DQ Node Volume and Achieved Bandwidth

  --- text_encoder ---
  Total graph nodes:    1417
  Q/DQ nodes:           1013  (71% of all nodes)
    DequantizeLinear:   611
    QuantizeLinear:     402
  Total elements:       155,735,580
  Total read+write:     914,871,816 bytes (0.915 GB)
  Activation Q/DQ dtypes: {'unknown': 804}
  Weight DQ dtypes:       {'uint16': 15, 'uint8': 97, 'int32': 97}
  Achieved BW (amd  ): 24.7 GB/s  (overhead=37ms, data=915MB)
  Achieved BW (intel): 32.6 GB/s  (overhead=28ms, data=915MB)
  Achieved BW (arm  ): 15.5 GB/s  (overhead=59ms, data=915MB)

  --- unet ---
  Total graph nodes:    5482
  Q/DQ nodes:           3937  (72% of all nodes)
    DequantizeLinear:   2389
    QuantizeLinear:     1548
  Total elements:       3,595,087,591
  Total read+write:     26,183,099,928 bytes (26.183 GB)
  Activation Q/DQ dtypes: {'unknown': 3096}
  Weight DQ dtypes:       {'uint8': 391, 'int32': 295, 'uint16': 155}
  Achieved BW (amd  ): 24.6 GB/s  (overhead=1065ms

In [8]:
# G.4: Compact summary with achieved bandwidth
for comp in COMPONENT_ORDER:
    qi = results["qdq"][comp]
    fi = results["float"][comp]
    dq = qi["qdq_details"]["DequantizeLinear"]["count"]
    q = qi["qdq_details"]["QuantizeLinear"]["count"]
    ratio = qi["qdq_node_count"] / fi["total_nodes"]
    bw_parts = []
    for dev in DEVICE_ORDER:
        dd = df[df["device"] == dev]
        f_ms = dd[(dd["model_type"]=="float")&(dd["component"]==comp)]["mean_ms"].values[0]
        q_ms = dd[(dd["model_type"]=="qdq")&(dd["component"]==comp)]["mean_ms"].values[0]
        oh = q_ms - f_ms
        bw = (qi["qdq_total_bytes"]/1e9)/(oh/1000) if oh > 0 and qi["qdq_total_bytes"] > 0 else 0
        bw_parts.append(f"{dev}={bw:.1f}")
    print(f"{comp}: DQ={dq} Q={q} total={qi['qdq_node_count']} "
          f"data={qi['qdq_total_bytes']/1e6:.0f}MB "
          f"params_f={fi['param_bytes']/1e6:.0f}MB params_q={qi['param_bytes']/1e6:.0f}MB "
          f"({qi['param_bytes']/fi['param_bytes']:.2f}x) "
          f"BW(GB/s): {', '.join(bw_parts)}")

text_encoder: DQ=611 Q=402 total=1013 data=915MB params_f=492MB params_q=162MB (0.33x) BW(GB/s): amd=24.7, intel=32.6, arm=15.5
unet: DQ=2389 Q=1548 total=3937 data=26183MB params_f=3438MB params_q=861MB (0.25x) BW(GB/s): amd=24.6, intel=14.3, arm=33.0
vae_decoder: DQ=488 Q=287 total=775 data=33238MB params_f=198MB params_q=50MB (0.25x) BW(GB/s): amd=16.1, intel=14.8, arm=18.2
vae_encoder: DQ=378 Q=223 total=601 data=18030MB params_f=137MB params_q=34MB (0.25x) BW(GB/s): amd=24.4, intel=11.4, arm=16.3


### G — Observations

**1. Q/DQ volume is massive:**
- unet: **3,937** Q/DQ nodes (DQ=2,389, Q=1,548), processing **26 GB** of tensor data per inference
- vae_decoder: 775 Q/DQ nodes processing **33 GB** (largest — high-resolution spatial feature maps)
- vae_encoder: 601 Q/DQ nodes processing **18 GB**
- text_encoder: 1,013 Q/DQ nodes processing **915 MB**
- **Total: 6,326 Q/DQ nodes, ~78 GB of data per inference step** — this is an enormous amount of elementwise work on top of the float compute

**2. Parameter compression is ~4× as expected for uint8 quantization:**
- Float total: 4,265 MB → QDQ total: 1,106 MB (0.26×)
- Consistent across components (0.25× for unet/vae, 0.33× for text_encoder)
- Note: this is disk/memory size of weights only. Runtime memory is dominated by activations.

**3. Achieved bandwidth of Q/DQ ops is reasonable (11–33 GB/s):**
| | AMD | Intel | ARM |
|-|-----|-------|-----|
| text_encoder | 24.7 | 32.6 | 15.5 |
| unet | 24.6 | 14.3 | 33.0 |
| vae_decoder | 16.1 | 14.8 | 18.2 |
| vae_encoder | 24.4 | 11.4 | 16.3 |

- Peak DRAM BW: AMD DDR5 ~40–50 GB/s, Intel/ARM LPDDR5x ~50–70 GB/s
- Achieved 30–65% of peak — **typical for many small kernel launches** with per-node allocation overhead
- This suggests the Q/DQ kernel itself is reasonably vectorized; the gap to peak is likely launch overhead, allocation, and cache misses from processing thousands of independent small tensors

**4. The high overhead in Section F is now explained by volume, not kernel inefficiency:**
- The 0.25–2.6× overhead/float fractions are consistent with processing **78 GB of Q/DQ data** on top of the float compute
- The irregular cross-device scaling (Section F.3–F.4) is consistent with workloads of different sizes hitting different cache/allocation bottlenecks

**5. Activation dtype info is missing from value_info** — ONNX graph doesn't have shape/type annotations for intermediate tensors (shown as `unknown`). We know from the quantization config that activations are uint16 per-tensor.

### Key Takeaways

| # | Finding | Impact |
|---|---------|--------|
| 1 | **QDQ without fusion is strictly slower than float** — Q/DQ nodes add overhead on top of identical fp32 compute | EXPECTED — not a bug |
| 2 | **Root cause: uint16 activations block all QDQ fusion** — ORT CPU EP integer kernels (`QLinearConv`, `QLinearMatMul`) only support int8/uint8. No CPU has native uint16×uint8 SIMD multiply. | ROOT CAUSE — explains 100% of the overhead |
| 3 | **uint8 weights + int32 bias are fusible in principle** — but activation type must also be int8/uint8 for the full `DQ→Op→Q` pattern to match | INFO — weights are not the blocker |
| 4 | **Re-quantizing with uint8 activations would enable fusion** — but may degrade diffusion output quality | ACTIONABLE — needs accuracy validation |
| 5 | **Weight-only quantization avoids the problem entirely** — no activation Q/DQ, only weight DQ that can fuse to `MatMulNBits` for linear layers | RECOMMENDED for CPU EP |
| 6 | **AMD is fastest overall**, ARM narrows gap under QDQ (single-threaded Q/DQ equalizes devices) | INFO |
| 7 | **Measurement variance is acceptable** — only text_encoder (lightweight) shows high CV | INFO — results are reliable |

**Bottom line:** These SD QDQ models with uint16 activations will **never** be faster than float on CPU EP. The quantization format is designed for EPs with native uint16 integer compute (e.g., QNN/NPU). For CPU inference, either re-quantize with uint8 activations to enable fusion, use weight-only quantization, or keep the float model.